# Staged Prediction Modeling
## Postoperative Complications After Lung Cancer Surgery — DLCAS-2026

---

**Modeling strategy (3 stages):**

| Stage | Goal | Method |
|---|---|---|
| 1 | Feature selection | LASSO Logistic Regression (L1) |
| 2 | Model comparison | LR, Random Forest, XGBoost — 5-fold CV |
| 3 | Final evaluation | Best model(s) on held-out test set |

**Outcome:** `compl` — any postoperative complication within 30 days (31.5% prevalence)

**Primary metric:** AUC-ROC  
**Secondary metrics:** Average Precision, Brier Score, Calibration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import os

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, brier_score_loss,
    confusion_matrix, classification_report, balanced_accuracy_score
)
from sklearn.calibration import calibration_curve

try:
    from xgboost import XGBClassifier
except ImportError:
    raise ImportError("pip install xgboost")

try:
    import shap
except ImportError:
    raise ImportError("pip install shap")

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

# ── CONFIG ────────────────────────────────────────────────────
# NOTE: preprocessed data is not included in this public repo.
# This must match OUTPUT_DIR from 01_preprocessing.ipynb (run that notebook
# first to generate X_train.csv / X_test.csv / y_train.csv / y_test.csv).
DATA_DIR    = "../../data/lung_processed"
RANDOM_SEED = 42
N_FOLDS     = 5

# Near-zero variance features to drop before modeling
# pet=97% positive, tnm_m=98% M0 — both mode-imputed to near-constant
# alcohol was excluded at preprocessing stage (95.3% missing — not imputed)
DROP_FEATURES = ['pet', 'tnm_m']

CV = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

In [ ]:
# ── Load preprocessed data ────────────────────────────────────
X_train = pd.read_csv(os.path.join(DATA_DIR, 'X_train.csv'))
X_test  = pd.read_csv(os.path.join(DATA_DIR, 'X_test.csv'))
y_train = pd.read_csv(os.path.join(DATA_DIR, 'y_train.csv')).squeeze()
y_test  = pd.read_csv(os.path.join(DATA_DIR, 'y_test.csv')).squeeze()

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Outcome balance — Train: {y_train.mean()*100:.1f}% | Test: {y_test.mean()*100:.1f}%")
print(f"\nFeatures loaded ({len(X_train.columns)}): {list(X_train.columns)}")

In [ ]:
# ── Pre-modeling checks ───────────────────────────────────────
# Drop near-zero variance features (identified in preprocessing analysis)
DROP_PRESENT = [f for f in DROP_FEATURES if f in X_train.columns]
X_train = X_train.drop(columns=DROP_PRESENT)
X_test  = X_test.drop(columns=DROP_PRESENT)
print(f"Dropped near-zero variance features: {DROP_PRESENT}")
print(f"Reason: mode-imputed to near-constant (pet=97% positive, alcohol=97% mode, tnm_m=98% M0)")
print(f"\nFinal feature count: {X_train.shape[1]}")
print(f"Features: {list(X_train.columns)}")

# Variance check
print("\nVariance check (flag if < 0.01):")
low_var = X_train.var()[X_train.var() < 0.01]
if len(low_var) > 0:
    print(low_var)
else:
    print("  All remaining features have variance >= 0.01")

FEATURE_NAMES = list(X_train.columns)

---
## Stage 1 — LASSO Feature Selection

Logistic regression with L1 (LASSO) penalty shrinks coefficients of weak predictors to zero.
We use cross-validated C selection (inverse regularization strength) to find the optimal penalty.

Features with non-zero coefficients at the optimal C are retained for Stage 2.

> **Note:** Features are standardized (z-score) before LASSO so coefficients are comparable.

In [ ]:
# Standardize features for LASSO (tree models do not need this)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=FEATURE_NAMES, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=FEATURE_NAMES, index=X_test.index
)
print("Features standardized (mean=0, std=1) — used for LR and LASSO only.")

In [ ]:
# ── LASSO with cross-validated C selection ────────────────────
lasso_cv = LogisticRegressionCV(
    Cs=np.logspace(-3, 2, 30),
    cv=CV,
    penalty='l1',
    solver='liblinear',
    scoring='roc_auc',
    max_iter=2000,
    random_state=RANDOM_SEED
)
lasso_cv.fit(X_train_scaled, y_train)

best_C = lasso_cv.C_[0]
print(f"Optimal C (LASSO): {best_C:.4f}  (log scale: {np.log10(best_C):.2f})")

# Coefficients
coef_df = pd.DataFrame({
    'Feature':     FEATURE_NAMES,
    'Coefficient': lasso_cv.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

selected   = coef_df[coef_df['Coefficient'] != 0]
eliminated = coef_df[coef_df['Coefficient'] == 0]

print(f"\nFeatures selected (coef ≠ 0): {len(selected)} / {len(FEATURE_NAMES)}")
print(f"Features eliminated (coef = 0): {len(eliminated)}")
print(f"\nEliminated: {eliminated['Feature'].tolist()}")

print("\nSelected features with coefficients:")
print(selected.to_string(index=False))

In [ ]:
# ── LASSO coefficient plot ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, max(5, len(selected) * 0.4)))

plot_df = selected.sort_values('Coefficient')
colors  = ['#E07050' if c < 0 else '#4CAF50' for c in plot_df['Coefficient']]
ax.barh(plot_df['Feature'], plot_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardised Coefficient (LASSO)', fontsize=12)
ax.set_title('LASSO Feature Selection — Non-zero Coefficients', fontweight='bold', fontsize=13)
ax.text(0.98, 0.02, f'C = {best_C:.4f}',
        transform=ax.transAxes, ha='right', fontsize=10, color='gray')
plt.tight_layout()
plt.show()

SELECTED_FEATURES = selected['Feature'].tolist()
print(f"\nSelected features for Stage 2: {SELECTED_FEATURES}")

---
## Stage 2 — Model Training & Cross-Validation

We train four models and compare them using stratified 5-fold cross-validation on the training set:

| Model | Features | Notes |
|---|---|---|
| Logistic Regression (full) | All features | Baseline linear model |
| Logistic Regression (LASSO-selected) | LASSO-selected only | Parsimonious linear model |
| Random Forest | All features | Non-linear ensemble |
| XGBoost | All features | Gradient boosting |

**Metrics reported:** AUC-ROC, Average Precision, Brier Score, Balanced Accuracy

In [ ]:
# ── Model definitions ─────────────────────────────────────────
models = {
    'LR (full)': (
        LogisticRegression(max_iter=2000, random_state=RANDOM_SEED),
        X_train_scaled, X_test_scaled
    ),
    'LR (LASSO)': (
        LogisticRegression(max_iter=2000, random_state=RANDOM_SEED),
        X_train_scaled[SELECTED_FEATURES], X_test_scaled[SELECTED_FEATURES]
    ),
    'Random Forest': (
        RandomForestClassifier(
            n_estimators=500, max_depth=6,
            min_samples_leaf=15, random_state=RANDOM_SEED, n_jobs=-1
        ),
        X_train, X_test
    ),
    'XGBoost': (
        XGBClassifier(
            n_estimators=300, max_depth=4,
            learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.8, min_child_weight=10,
            random_state=RANDOM_SEED, eval_metric='logloss',
            verbosity=0
        ),
        X_train, X_test
    )
}

print("Models defined:")
for name, (model, Xtr, _) in models.items():
    print(f"  {name:<22} features: {Xtr.shape[1]}")

In [ ]:
# ── Stratified 5-fold cross-validation ───────────────────────
from sklearn.metrics import make_scorer

scoring = {
    'roc_auc':           'roc_auc',
    'avg_precision':     'average_precision',
    'balanced_accuracy': 'balanced_accuracy',
    'brier':             make_scorer(brier_score_loss, response_method='predict_proba',
                                     greater_is_better=False)
}

cv_results = {}
for name, (model, X_tr, _) in models.items():
    print(f"Running CV: {name}...", end=' ')
    res = cross_validate(model, X_tr, y_train, cv=CV,
                         scoring=scoring, return_train_score=False)
    cv_results[name] = res
    print(f"AUC = {res['test_roc_auc'].mean():.3f} ± {res['test_roc_auc'].std():.3f}")

print("\nCross-validation complete.")

In [ ]:
# ── CV results table ──────────────────────────────────────────
rows = []
for name, res in cv_results.items():
    rows.append({
        'Model':              name,
        'AUC-ROC':            f"{res['test_roc_auc'].mean():.3f} ± {res['test_roc_auc'].std():.3f}",
        'Avg Precision':      f"{res['test_avg_precision'].mean():.3f} ± {res['test_avg_precision'].std():.3f}",
        'Balanced Acc':       f"{res['test_balanced_accuracy'].mean():.3f} ± {res['test_balanced_accuracy'].std():.3f}",
        'Brier Score':        f"{-res['test_brier'].mean():.3f} ± {res['test_brier'].std():.3f}",
        'AUC (raw)':          res['test_roc_auc'].mean()
    })

cv_table = pd.DataFrame(rows).sort_values('AUC (raw)', ascending=False)
print("5-Fold Cross-Validation Results (mean ± std):")
print("=" * 90)
print(cv_table.drop(columns='AUC (raw)').to_string(index=False))
print("=" * 90)
print("\nLower Brier Score = better calibration (perfect = 0, null = prevalence² = ~0.215)")

In [ ]:
# ── CV AUC boxplot ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = list(cv_results.keys())
auc_data    = [cv_results[n]['test_roc_auc'] for n in model_names]
ap_data     = [cv_results[n]['test_avg_precision'] for n in model_names]

bp1 = axes[0].boxplot(auc_data, labels=model_names, patch_artist=True, notch=False)
for patch, color in zip(bp1['boxes'], ['#88BBDD', '#5599CC', '#E07050', '#4CAF50']):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[0].set_ylabel('AUC-ROC')
axes[0].set_title('5-Fold CV: AUC-ROC', fontweight='bold', fontsize=13)
axes[0].axhline(0.7, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='AUC = 0.70')
axes[0].legend()
plt.setp(axes[0].get_xticklabels(), rotation=15, ha='right')

bp2 = axes[1].boxplot(ap_data, labels=model_names, patch_artist=True)
for patch, color in zip(bp2['boxes'], ['#88BBDD', '#5599CC', '#E07050', '#4CAF50']):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[1].set_ylabel('Average Precision')
axes[1].set_title('5-Fold CV: Average Precision', fontweight='bold', fontsize=13)
axes[1].axhline(0.315, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Baseline (prevalence)')
axes[1].legend()
plt.setp(axes[1].get_xticklabels(), rotation=15, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# ── ROC curves on full training set (refit each model) ────────
fig, ax = plt.subplots(figsize=(8, 7))
colors_roc = ['#88BBDD', '#2266AA', '#E07050', '#4CAF50']

fitted_models = {}
for (name, (model, X_tr, X_te)), color in zip(models.items(), colors_roc):
    model.fit(X_tr, y_train)
    fitted_models[name] = (model, X_tr, X_te)
    proba = model.predict_proba(X_tr)[:, 1]
    fpr, tpr, _ = roc_curve(y_train, proba)
    auc = roc_auc_score(y_train, proba)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name}  (train AUC={auc:.3f})')

ax.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Training Set (refit)', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()
print("Note: train AUC is optimistic — use CV results for unbiased comparison.")

---
## SHAP Feature Importance (XGBoost)

SHAP (SHapley Additive exPlanations) decomposes each prediction into feature contributions.

- **Beeswarm plot**: each dot = one patient; position = SHAP value; color = feature value
- **Bar plot**: mean |SHAP| = average absolute contribution across all patients

SHAP values are in log-odds units: positive = increases predicted risk of complication.

In [ ]:
xgb_model = fitted_models['XGBoost'][0]
X_shap    = fitted_models['XGBoost'][1]  # training set, unscaled

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_shap)

# ── Beeswarm plot ─────────────────────────────────────────────
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_shap, plot_type='dot',
                  feature_names=FEATURE_NAMES, show=False, max_display=20)
plt.title('SHAP Beeswarm — XGBoost (training set)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# ── Bar plot (mean |SHAP|) ────────────────────────────────────
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_shap, plot_type='bar',
                  feature_names=FEATURE_NAMES, show=False, max_display=20)
plt.title('SHAP Feature Importance — XGBoost', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP importance table ─────────────────────────────────────
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_table = pd.DataFrame({
    'Feature':        FEATURE_NAMES,
    'Mean |SHAP|':    mean_abs_shap,
    'Direction':      ['↑ Risk' if shap_values[:, i].mean() > 0
                       else '↓ Risk' for i in range(len(FEATURE_NAMES))]
}).sort_values('Mean |SHAP|', ascending=False)

print("SHAP Importance Table (XGBoost, training set):")
print(shap_table.to_string(index=False))

---
## Stage 3 — Final Evaluation on Held-Out Test Set

The test set (20%, N=239) was **not used at any point** during training or CV.
These results represent the unbiased estimate of model performance on new patients.

In [ ]:
# ── Test set metrics ──────────────────────────────────────────
test_rows = []
test_probas = {}

for name, (model, _, X_te) in fitted_models.items():
    proba = model.predict_proba(X_te)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    test_probas[name] = proba

    auc    = roc_auc_score(y_test, proba)
    ap     = average_precision_score(y_test, proba)
    brier  = brier_score_loss(y_test, proba)
    bal_acc = balanced_accuracy_score(y_test, pred)

    test_rows.append({
        'Model':          name,
        'AUC-ROC':        round(auc,    3),
        'Avg Precision':  round(ap,     3),
        'Brier Score':    round(brier,  3),
        'Balanced Acc':   round(bal_acc, 3)
    })

test_table = pd.DataFrame(test_rows).sort_values('AUC-ROC', ascending=False)
print("Test Set Performance (N=239, held-out):")
print("=" * 70)
print(test_table.to_string(index=False))
print("=" * 70)
print(f"\nNull model Brier score (predict prevalence): {(y_test.mean()*(1-y_test.mean())):.3f}")

In [ ]:
# ── ROC curves — test set ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors_roc = ['#88BBDD', '#2266AA', '#E07050', '#4CAF50']
for (name, proba), color in zip(test_probas.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, color=color, linewidth=2,
                 label=f'{name}  AUC={auc:.3f}')

axes[0].plot([0,1],[0,1],'k--', linewidth=1, alpha=0.4, label='Random')
axes[0].fill_between([0,1],[0,1], alpha=0.05, color='gray')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curves — Test Set', fontweight='bold', fontsize=13)
axes[0].legend(fontsize=9)

# Precision-Recall curves
for (name, proba), color in zip(test_probas.items(), colors_roc):
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[1].plot(rec, prec, color=color, linewidth=2,
                 label=f'{name}  AP={ap:.3f}')

axes[1].axhline(y_test.mean(), color='gray', linestyle='--',
                linewidth=1, label=f'Baseline (prevalence={y_test.mean():.2f})')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curves — Test Set', fontweight='bold', fontsize=13)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Calibration curves — test set ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for (name, proba), color in zip(test_probas.items(), colors_roc):
    frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=8)
    axes[0].plot(mean_pred, frac_pos, 'o-', color=color,
                 linewidth=2, markersize=6, label=name)

axes[0].plot([0,1],[0,1],'k--', linewidth=1, alpha=0.6, label='Perfect calibration')
axes[0].set_xlabel('Mean Predicted Probability', fontsize=12)
axes[0].set_ylabel('Fraction of Positives', fontsize=12)
axes[0].set_title('Calibration Curves — Test Set', fontweight='bold', fontsize=13)
axes[0].legend(fontsize=9)

# Brier score bar chart
brier_scores = {name: brier_score_loss(y_test, proba)
                for name, proba in test_probas.items()}
null_brier   = y_test.mean() * (1 - y_test.mean())

bar_colors = ['#4CAF50' if v < null_brier else '#E07050'
              for v in brier_scores.values()]
axes[1].bar(brier_scores.keys(), brier_scores.values(),
            color=bar_colors, edgecolor='white', alpha=0.8)
axes[1].axhline(null_brier, color='red', linestyle='--',
                linewidth=1.5, label=f'Null model ({null_brier:.3f})')
axes[1].set_ylabel('Brier Score (lower = better)', fontsize=12)
axes[1].set_title('Brier Scores — Test Set', fontweight='bold', fontsize=13)
axes[1].legend()
plt.setp(axes[1].get_xticklabels(), rotation=15, ha='right')
for i, (name, v) in enumerate(brier_scores.items()):
    axes[1].text(i, v + 0.002, f'{v:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrices at 0.5 threshold ───────────────────────
fig, axes = plt.subplots(1, len(fitted_models), figsize=(16, 4))

for ax, (name, proba) in zip(axes, test_probas.items()):
    pred = (proba >= 0.5).astype(int)
    cm   = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=['Pred 0', 'Pred 1'],
                yticklabels=['True 0', 'True 1'],
                annot_kws={'size': 13})
    auc = roc_auc_score(y_test, proba)
    ax.set_title(f'{name}\nAUC={auc:.3f}', fontweight='bold', fontsize=11)

plt.suptitle('Confusion Matrices — Test Set (threshold = 0.50)',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print("\nNote: threshold = 0.50 is the default. Optimal threshold may differ.")

In [ ]:
# ── Optimal threshold (Youden's J = sensitivity + specificity - 1) ─
print("Optimal thresholds by Youden's J index (test set):")
print("=" * 65)
print(f"{'Model':<22} {'Threshold':>10} {'Sensitivity':>13} {'Specificity':>13} {'PPV':>8} {'NPV':>8}")
print("=" * 65)

for name, proba in test_probas.items():
    fpr, tpr, thresholds = roc_curve(y_test, proba)
    j_scores = tpr - fpr
    best_idx  = np.argmax(j_scores)
    best_thr  = thresholds[best_idx]
    pred_opt  = (proba >= best_thr).astype(int)
    cm        = confusion_matrix(y_test, pred_opt)
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    ppv  = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv  = tn / (tn + fn) if (tn + fn) > 0 else 0
    print(f"{name:<22} {best_thr:>10.3f} {sens:>12.1%} {spec:>12.1%} {ppv:>7.1%} {npv:>7.1%}")

print("=" * 65)

In [ ]:
# ── Random Forest feature importance (comparison with SHAP) ───
rf_model = fitted_models['Random Forest'][0]
rf_imp   = pd.DataFrame({
    'Feature':    FEATURE_NAMES,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, max(5, len(FEATURE_NAMES) * 0.38)))
bars = ax.barh(rf_imp['Feature'], rf_imp['Importance'],
               color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)', fontsize=12)
ax.set_title('Random Forest Feature Importance', fontweight='bold', fontsize=13)
for bar, val in zip(bars, rf_imp['Importance']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Final summary table ───────────────────────────────────────
print("FINAL PERFORMANCE SUMMARY")
print("=" * 75)
print(f"{'Model':<22} {'CV AUC':>10} {'Test AUC':>10} {'Test AP':>10} {'Test Brier':>12}")
print("=" * 75)

for name in cv_results:
    cv_auc   = cv_results[name]['test_roc_auc'].mean()
    test_row = next(r for r in test_rows if r['Model'] == name)
    print(f"{name:<22} {cv_auc:>10.3f} {test_row['AUC-ROC']:>10.3f} "
          f"{test_row['Avg Precision']:>10.3f} {test_row['Brier Score']:>12.3f}")

print("=" * 75)
print(f"\nOutcome prevalence: {y_test.mean():.1%}")
print(f"Null Brier score: {y_test.mean()*(1-y_test.mean()):.3f}")
print(f"Train N={len(y_train)}, Test N={len(y_test)}, Total N={len(y_train)+len(y_test)}")

---
## Conclusions

### Interpretation guide

| AUC-ROC | Interpretation |
|---|---|
| 0.50 | No better than chance |
| 0.60–0.70 | Poor discrimination |
| 0.70–0.80 | Acceptable |
| 0.80–0.90 | Good |
| > 0.90 | Excellent (may indicate overfitting) |

### Known limitations of this model

| Issue | Impact |
|---|---|
| CCI 62% missing → median-imputed | Comorbidity signal is diluted |
| BMI 44% missing → median-imputed | May underestimate BMI effect |
| TNM staging 45% missing → mode-imputed | Staging features may be noisy |
| No neoadjuvant treatment data | Potentially important predictor absent |
| `compl` only — subtype outcomes unavailable | Cannot model specific complication types |

### Recommended next steps
1. **Obtain complete comorbidity data** (comorbiditeiten dataset) — would substantially improve CCI quality
2. **Sensitivity analysis**: re-run on subset with complete CCI + BMI + TNM (~300 patients)
3. **Calibration**: if Brier score or calibration curve shows systematic over/under-prediction → apply Platt scaling
4. **External validation**: validate on a held-out hospital or year cohort before clinical use
5. **TRIPOD reporting**: document model development following TRIPOD-AI checklist